# 04. Baseline Federated Learning Analysis (ANSI/AAMI EC57)
## Master Thesis: Resource-Adaptive Federated Kolmogorov-Arnold Networks

### Executive Summary & Scientific Context
This notebook analyzes and visualizes experimental results from the centralized and federated baseline evaluations on the **MIT-BIH Arrhythmia Database**:
- **Dataset Partitioning**: 48 patient recordings partitioned strictly into patient-disjoint train (30 patients, 51,020 beats), validation (8 patients, 17,998 beats), and test sets (10 patients, 18,175 beats) per ANSI/AAMI EC57 standards.
- **Federated Dirichlet Skew**: Simulates heterogeneous edge environments across $\alpha \in \{0.1, 0.5, 1.0\}$ across $N = 8$ clients.
- **Baseline Models**:
  1. **FedAvg-MLP**: Multi-Layer Perceptron parameter-matched baseline ($5,477$ parameters per node).
  2. **FedAvg-KAN**: Fast Kolmogorov-Arnold Network with fixed uniform grid ($G=5, k=3$, $5,349$ parameters per node).
  3. **FedProx-KAN**: FastKAN regularized with proximal parameter penalty ($\mu = 0.01$).


In [ ]:
import os
import sys
from pathlib import Path

# Set project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
print(f"Working directory: {PROJECT_ROOT}")


---
### 1. Load Experimental Baseline Results
We load the master consolidated summary and individual communication round trajectories across all models and skew levels.


In [ ]:
summary_csv = PROJECT_ROOT / "results/baselines/baseline_comparison_summary.csv"
df_baselines = pd.read_csv(summary_csv)
print(f"Loaded {len(df_baselines)} experimental baseline configurations:")
display(df_baselines)


---
### 2. Accuracy & Macro F1 Trajectories Across Non-IID Skews ($\alpha \in \{0.1, 0.5, 1.0\}$)
Below we load the per-round evaluation traces on the held-out patient-level test split (18,175 heartbeats).


In [ ]:
alphas = [0.1, 0.5, 1.0]
models = ["FedAvg-MLP", "FedAvg-KAN", "FedProx-KAN"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for idx, alpha in enumerate(alphas):
    ax = axes[idx]
    for model in models:
        metrics_file = PROJECT_ROOT / f"results/baselines/metrics_{model}_alpha_{alpha}.csv"
        if metrics_file.exists():
            df_m = pd.read_csv(metrics_file)
            ax.plot(df_m["round"], df_m["test_accuracy"] * 100.0, marker="o", linewidth=2.0, label=f"{model} (Acc)")
            ax.plot(df_m["round"], df_m["test_macro_f1"] * 100.0, linestyle="--", alpha=0.7, label=f"{model} (F1)")
            
    ax.set_title(f"Dirichlet $\\alpha = {alpha}$ ({'Severe Skew' if alpha == 0.1 else 'Moderate Skew' if alpha == 0.5 else 'Uniform-Like'})", fontsize=11, fontweight="bold")
    ax.set_xlabel("Federated Communication Round", fontsize=10)
    if idx == 0:
        ax.set_ylabel("Metric (%)", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.6)
    if idx == 0:
        ax.legend(loc="lower right", fontsize=8)

fig.suptitle("Convergence Dynamics on Patient-Disjoint Test Set (18,175 Beats)", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()


---
### 3. ANSI/AAMI EC57 Class-Wise F1 Retention Under Non-IID Skew
MIT-BIH Arrhythmia Database exhibits severe natural class imbalance:
- **Class N (Normal)**: ~82.7% of all beats.
- **Class SVEB (Supraventricular Ectopic)**: ~2.5%.
- **Class VEB (Ventricular Ectopic)**: ~6.5%.
- **Class F (Fusion)**: ~0.7%.
- **Class Q (Unknown / Paced)**: ~7.6%.

In federated non-IID settings, naive aggregation can result in catastrophic forgetting of minority classes.


In [ ]:
f1_records = []
for alpha in alphas:
    for model in models:
        json_file = PROJECT_ROOT / f"results/baselines/summary_{model}_alpha_{alpha}.json"
        if json_file.exists():
            with open(json_file) as f:
                data = json.load(f)
            class_f1 = data.get("per_class_f1", {})
            for c, score in class_f1.items():
                f1_records.append({
                    "Model": model,
                    "Alpha": alpha,
                    "AAMI Class": c,
                    "F1 Score (%)": score * 100.0,
                })

df_f1 = pd.DataFrame(f1_records)

# Plot class-wise comparison for severe non-IID skew (alpha = 0.1)
plt.figure(figsize=(10, 5))
df_f1_skewed = df_f1[df_f1["Alpha"] == 0.1]
sns.barplot(data=df_f1_skewed, x="AAMI Class", y="F1 Score (%)", hue="Model", palette=["#e74c3c", "#3498db", "#2ecc71"])
plt.title("ANSI/AAMI EC57 Minority Class F1 Preservation under Severe Skew ($\alpha = 0.1$)", fontsize=12, fontweight="bold")
plt.ylabel("F1-Score (%)", fontsize=11)
plt.ylim(0, 100)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(framealpha=0.9)
plt.tight_layout()
plt.show()


---
### 4. Communication Cost & Parameter Footprint
Comparing parameter budget per client and communication bandwidth requirements:
- **MLP Baseline**: $32 \times 128 + 128 \times 5 + 128 + 5 = 5,477$ parameters.
- **Fixed FastKAN Baseline**: Layer 1 $(32 \to 16, G=5, k=3)$, Layer 2 $(16 \to 5, G=5, k=3) = 5,349$ parameters.
- **Communication per Round**: $5,349 \times 8 \text{ clients} = 42,792$ parameters.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

sns.barplot(data=df_baselines, x="Algorithm", y="Final Accuracy", hue="Alpha", ax=ax1, palette="Blues")
ax1.set_title("Global Test Accuracy by Skew Level", fontweight="bold", fontsize=11)
ax1.set_ylabel("Accuracy", fontsize=10)
ax1.set_ylim(0.7, 0.9)

sns.barplot(data=df_baselines, x="Algorithm", y="Comm (MB)", ax=ax2, color="#34495e")
ax2.set_title("Total Communication Volume (MB)", fontweight="bold", fontsize=11)
ax2.set_ylabel("Megabytes (MB)", fontsize=10)

plt.tight_layout()
plt.show()


---
### 5. Key Baseline Conclusions & Motivation for Adaptive KANs
1. **Spline Representation Advantage**: FastKAN achieves consistently superior accuracy ($86.83\%$ vs $75.66\%$ at $\alpha=0.1$) compared to MLP under identical parameter budget ($~5.3\text{k}$ params).
2. **Fixed-Grid Limitation**: In a fixed $(G=5, k=3)$ architecture, all 8 edge nodes transmit 5,349 parameters regardless of whether their local ECG distribution is simple or clinically complex.
3. **Motivation**: This motivates the **Reinforcement Learning Meta-Controller (PPO)** to dynamically assign $(G_i^t, k_i^t)$ based on real-time client curvature, bandwidth, and loss telemetry.
